# 🌐 Extended Providers — AWS Bedrock + GCP Gemini + Anthropic Claude

## Onboard 3 non-Microsoft-Foundry LLM backends through Citadel Governance Hub

This notebook **extends** `llm-backend-onboarding-runner.ipynb`. The base notebook provisions
Microsoft Foundry models (managed-identity auth). This one keeps that pipeline and **adds three
non-Azure backends** authenticated with simple API keys, then validates them through both LLM
API surfaces of the gateway.

### APIM API URL suffixes (the part right after `https://<apim-gateway>/`)

| API in APIM | URL suffix | Then operation/path |
|---|---|---|
| Azure OpenAI API | `openai` | `/deployments/{model}/chat/completions?api-version=...` |
| Universal LLM API | `models` | `/chat/completions`, `/deployments`, `/embeddings`, ... |
| Unified AI API (wildcard) | `unified-ai` | `/v1/chat/completions`, `/bedrock/...`, `/gemini/...`, `/claude/...`, `/deployments` |

### Two LLM APIs surfaced by the gateway and used in this notebook

| API | Inbound path | Purpose | Backends covered by this notebook |
|---|---|---|---|
| **Universal LLM API** | `/models/chat/completions` | OpenAI-compat only | Microsoft Foundry, Bedrock-OpenAI-compat, Gemini-OpenAI-compat |
| **Unified AI API — OpenAI-compat** | `/unified-ai/v1/chat/completions` | OpenAI-compat across all OpenAI-compat-capable backends | Microsoft Foundry, Bedrock-OpenAI-compat, Gemini-OpenAI-compat |
| **Unified AI API — `/bedrock/`** | `/unified-ai/bedrock/model/{modelId}/converse` | **Native** AWS Bedrock (Converse / InvokeModel) | `aws-bedrock` (api-key-bearer) |
| **Unified AI API — `/gemini/`** | `/unified-ai/gemini/v1beta/models/{model}:generateContent` | **Native** GCP Gemini | `gemini` (api-key-gemini) |
| **Unified AI API — `/claude/`** | `/unified-ai/claude/v1/messages` | **Native** Anthropic Messages | `anthropic` (api-key-anthropic) |

### Backend-type matrix

| Backend type | Auth (this notebook) | OpenAI-compat path | Native path prefix |
|---|---|---|---|
| `ai-foundry` | managed-identity | `/chat/completions` (Universal LLM) and `/v1/chat/completions` (Unified AI) | n/a |
| `aws-bedrock` | `api-key-bearer` (Bedrock long-lived API key) | n/a | `/bedrock/model/{modelId}/converse` |
| `aws-bedrock-mantle` | `api-key-bearer` | `/chat/completions` (Universal LLM) and `/v1/chat/completions` (Unified AI) | n/a |
| `gemini` | `api-key-gemini` (`x-goog-api-key`) | n/a | `/gemini/v1beta/models/{model}:generateContent` |
| `gemini-openai` | `api-key-bearer` | `/chat/completions` (Universal LLM) and `/v1/chat/completions` (Unified AI) | n/a |
| `anthropic` | `api-key-anthropic` (`x-api-key` + `anthropic-version`) | n/a | `/claude/v1/messages` |

### Gateway changes shipped with this notebook

- **New backend type** `gemini` and **new auth type** `api-key-gemini` (`x-goog-api-key`).
- **New auth type** `api-key-anthropic` (`x-api-key` + `anthropic-version`) and **named value** `anthropic-version`.
- **Native-prefix api-types**: `bedrock-native` (`/bedrock`), `gemini-native` (`/gemini`), `claude-native` (`/claude`) in `frag-metadata-config.xml`.
- **`compatible-pool-types`** filter in `frag-request-processor.xml` + `frag-set-target-backend-pool.xml` so each native prefix only matches its native pool type and `/chat/completions` / `/v1/chat/completions` only match OpenAI-compat-capable pool types — **no model-name suffix tricks needed**.
- **Prefix-strip path-builder** branches for `bedrock-native` / `gemini-native` (forward path unchanged) and a fixed `/v1/messages` rewrite for `claude-native`.
- **Native APIM Backend authorization**: api-key auth headers are configured directly on the APIM `Backend` resource via `credentials.header`, not in policy fragments. APIM resolves `{{namedValueKey}}` to the named value's secret (or Key Vault reference) at backend create/update time.

### Prerequisites

1. An existing Citadel Governance Hub deployment (APIM + managed identity provisioned).
2. (Optional) An Azure Key Vault to hold non-Azure provider keys as secrets — recommended over inline `secretValue`.
3. **Provider API keys**:
   - **AWS Bedrock** API key (Bedrock console → Access Management → API keys). Long-lived bearer token.
   - **GCP Gemini** API key (https://aistudio.google.com/apikey).
   - **Anthropic** API key (https://console.anthropic.com → API Keys).

### ⚠️ Secret format — APIM Backend headers cannot concatenate strings

APIM resolves `{{namedValueKey}}` tokens on the **Backend resource** only when the entire header
value is the bare token. It does **not** substitute named values inside concatenated strings
(e.g. `Authorization: Bearer {{key}}` is sent literally to the upstream — APIM does not
interpolate `{{key}}` once a literal prefix is attached). To stay native and keep policies simple,
the **Key Vault secret (or inline named value) must contain the complete header value**:

| Auth type used by | Header set on backend | Secret value stored in Key Vault |
|---|---|---|
| `api-key-bearer` (Bedrock native + Mantle, Gemini-OpenAI-compat) | `Authorization: <secret>` | `Bearer sk-abc123…` |
| `api-key-header` | `api-key: <secret>` | `sk-abc123…` |
| `api-key-gemini` (Gemini native) | `x-goog-api-key: <secret>` | `sk-abc123…` |
| `api-key-anthropic` (Anthropic native) | `x-api-key: <secret>` (+ `anthropic-version` from a separate named value) | `sk-abc123…` |

**Practical impact**: a provider that has both a native surface and an OpenAI-compat surface
(e.g. Gemini) needs **two separate Key Vault secrets** — one storing the raw key (for
`x-goog-api-key`) and one storing `Bearer <key>` (for `Authorization`). The init cell below
defines a separate `authConfig` per backend so each backend points at its own named value /
KV URI; do **not** share named values across backends with different header conventions.



<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

This notebook ships with **REPLACE placeholders** for every credential. Tests automatically skip
when a model maps to a backend whose key is still `REPLACE_*` (no failures, no surprises).

- `init_from_azd = False` by default so the placeholders below are not overwritten by your `azd` env.
  Flip to `True` when you maintain `LLM_BACKEND_CONFIG` in `azd` and want to merge it with the new
  backends here.
- All three new backends use **simple API key auth** so you don't need AWS SigV4 or Workload
  Identity Federation just to validate connectivity.
- Each provider can take either an **inline secret value** (testing only — stored as an APIM
  named value) **or** a **Key Vault secret URI** (production-friendly, rotatable). The Bicep module
  picks Key Vault when `keyVaultSecretUri` is provided; otherwise it stores `secretValue` directly.


In [ ]:
import os
import sys, json, requests, time, uuid
from urllib.parse import quote
sys.path.insert(1, "../shared")  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

inference_api_version = "2024-05-01-preview"

# ============================================================================
# 🔧 INITIALIZATION MODE
# ============================================================================
# Set False to keep the REPLACE values in this notebook. Set True to merge with
# whatever `azd env get-value LLM_BACKEND_CONFIG` returns from your active azd env.
init_from_azd = True

# ============================================================================
# REQUIRED: Citadel Governance Hub coordinates
# ============================================================================
governance_hub_resource_group = "REPLACE"          # Resource group of the deployed Citadel Governance Hub
location                      = "REPLACE"          # Azure region of the deployment

# ============================================================================
# 🔑 PROVIDER CREDENTIALS — leave as REPLACE_* to skip a backend's tests
# ----------------------------------------------------------------------------
# IMPORTANT — APIM backend credentials.header substitutes `{{namedValueKey}}` only
# when it IS the entire header value (no concatenation). The Key Vault secret /
# inline named value must therefore contain the **complete header value** that
# APIM will send to the upstream:
#
#   Bearer auth (Authorization header)  →  secret value = "Bearer sk-abc..."
#   Direct key headers (x-api-key,
#       x-goog-api-key, api-key)        →  secret value = "sk-abc..."  (raw key)
#
# A provider with both a native and an OpenAI-compat backend (e.g. Gemini) needs
# TWO separate secrets — one raw, one prefixed — because they map to different
# header names (x-goog-api-key vs Authorization). Each backend below has its own
# *_inline / *_key_vault_uri pair so the secrets stay isolated.
# ============================================================================

# --- AWS Bedrock NATIVE  (Authorization: Bearer <secret>) -------------------
aws_bedrock_region                = "eu-north-1"
aws_bedrock_native_inline         = "REPLACE_AWS_BEDROCK_BEARER_VALUE"          # store full "Bearer sk-..." in the secret
aws_bedrock_native_key_vault_uri  = "https://kv-REPLACE.vault.azure.net/secrets/aws-key-bearer"

# --- AWS Bedrock OpenAI-compat (Mantle)  (Authorization: Bearer <secret>) ---
aws_bedrock_openai_inline         = "REPLACE_AWS_BEDROCK_BEARER_VALUE"          # may point at the same KV secret as native
aws_bedrock_openai_key_vault_uri  = "https://kv-REPLACE.vault.azure.net/secrets/aws-key-bearer"

# --- GCP Gemini NATIVE  (x-goog-api-key: <secret>) --------------------------
gemini_native_inline              = "REPLACE_GEMINI_RAW_KEY"                    # raw API key, no prefix
gemini_native_key_vault_uri       = "https://kv-REPLACE.vault.azure.net/secrets/gcp-key"

# --- GCP Gemini OpenAI-compat  (Authorization: Bearer <secret>) -------------
gemini_openai_inline              = "REPLACE_GEMINI_BEARER_VALUE"               # store full "Bearer sk-..." here — DIFFERENT secret
gemini_openai_key_vault_uri       = "https://kv-REPLACE.vault.azure.net/secrets/gcp-key-bearer"

# --- Anthropic Claude NATIVE  (x-api-key: <secret>) -------------------------
anthropic_inline                  = "REPLACE_ANTHROPIC_RAW_KEY"                 # raw API key, no prefix
anthropic_key_vault_uri           = "https://kv-REPLACE.vault.azure.net/secrets/anthropic-key"
anthropic_version                 = "2023-06-01"                                # value of the anthropic-version named value

# Optional Key Vault for the inline-secret backends. Required ONLY when at least one of the
# *_key_vault_uri values is set. The APIM managed identity must have `Key Vault Secrets User`
# role on this vault.
key_vault_name = "kv-REPLACE"

# ============================================================================
# Backend definitions — three native + two OpenAI-compat passthroughs
# ----------------------------------------------------------------------------
# Each backend has its own authConfig (no sharing) so the secret format on the
# named value matches what APIM will send literally on the wire.
# ============================================================================

def _auth_config(named_value_key: str, kv_uri: str, inline: str) -> dict:
    """Choose Key Vault reference when available, else inline secret value."""
    if kv_uri:
        return {"namedValueKey": named_value_key, "keyVaultSecretUri": kv_uri}
    return {"namedValueKey": named_value_key, "secretValue": inline}


def _is_placeholder(value: str) -> bool:
    return (not value) or value.startswith("REPLACE")


# Per-backend named-value keys (one per backend, no sharing)
NV_BEDROCK_NATIVE      = "aws-bedrock-native-extended-bearer"
NV_BEDROCK_OPENAI      = "aws-bedrock-openai-extended-bearer"
NV_GEMINI_NATIVE       = "gemini-native-extended-key"
NV_GEMINI_OPENAI       = "gemini-openai-extended-bearer"
NV_ANTHROPIC_NATIVE    = "anthropic-native-extended-key"

extended_provider_backends = []

# ----- 1. AWS Bedrock NATIVE (Converse / InvokeModel) -----------------------
# Secret value MUST be "Bearer <bedrock-api-key>" (full Authorization header value).
extended_provider_backends.append({
    "backendId": "bedrock-native-extended",
    "backendType": "aws-bedrock",
    "authType": "api-key-bearer",       # use Bedrock API key (Bearer) — simpler than SigV4 for validation
    "endpoint": f"https://bedrock-runtime.{aws_bedrock_region}.amazonaws.com",
    "priority": 1,
    "weight": 100,
    "authConfig": _auth_config(NV_BEDROCK_NATIVE, aws_bedrock_native_key_vault_uri, aws_bedrock_native_inline),
    "supportedModels": [
        {"name": "global.amazon.nova-2-lite-v1:0",  "modelFormat": "Amazon", "modelVersion": "1", "sku": "OnDemand", "capacity": 1, "retirementDate": "2099-12-30"},
        {"name": "amazon.nova-lite-v1:0", "modelFormat": "Amazon", "modelVersion": "1", "sku": "OnDemand", "capacity": 1, "retirementDate": "2099-12-30"},
    ],
})

# ----- 2. GCP Gemini NATIVE (generateContent / streamGenerateContent) ------
# Secret value MUST be the raw Gemini API key (used as `x-goog-api-key`).
extended_provider_backends.append({
    "backendId": "gemini-native-extended",
    "backendType": "gemini",
    "authType": "api-key-gemini",
    "endpoint": "https://generativelanguage.googleapis.com",
    "priority": 1,
    "weight": 100,
    "authConfig": _auth_config(NV_GEMINI_NATIVE, gemini_native_key_vault_uri, gemini_native_inline),
    "supportedModels": [
        {"name": "gemini-2.5-flash-lite", "modelFormat": "Google", "modelVersion": "1", "sku": "Standard", "capacity": 1, "retirementDate": "2099-12-30"},
        {"name": "gemini-2.5-flash",      "modelFormat": "Google", "modelVersion": "1", "sku": "Standard", "capacity": 1, "retirementDate": "2099-12-30"},
    ],
})

# ----- 3. Anthropic Claude NATIVE (Messages API) ---------------------------
# Secret value MUST be the raw Anthropic API key (used as `x-api-key`).
extended_provider_backends.append({
    "backendId": "anthropic-native-extended",
    "backendType": "anthropic",
    "authType": "api-key-anthropic",
    "endpoint": "https://api.anthropic.com",
    "priority": 1,
    "weight": 100,
    "authConfig": _auth_config(NV_ANTHROPIC_NATIVE, anthropic_key_vault_uri, anthropic_inline),
    "supportedModels": [
        {"name": "claude-sonnet-4-6", "modelFormat": "Anthropic", "modelVersion": "1", "sku": "Standard", "capacity": 1, "retirementDate": "2099-12-30"},
        {"name": "claude-haiku-4-5",  "modelFormat": "Anthropic", "modelVersion": "1", "sku": "Standard", "capacity": 1, "retirementDate": "2099-12-30"},
    ],
})

# ----- 4. AWS Bedrock OpenAI-compat (Mantle) — fans into /v1/chat/completions ----
# Bedrock's OpenAI-compatible endpoint also uses `Authorization: Bearer <key>`. It can
# point at the SAME Key Vault secret as the native Bedrock backend (the header value
# is identical for both). A separate named-value key is still used so the two backends
# can be rotated independently if you ever need to.
extended_provider_backends.append({
    "backendId": "bedrock-openai-compat-extended",
    "backendType": "aws-bedrock-mantle",
    "authType": "api-key-bearer",
    "endpoint": f"https://bedrock-mantle.{aws_bedrock_region}.api.aws",
    "priority": 1,
    "weight": 100,
    "authConfig": _auth_config(NV_BEDROCK_OPENAI, aws_bedrock_openai_key_vault_uri, aws_bedrock_openai_inline),
    "supportedModels": [
        # OpenAI-compat surface uses Bedrock's native model-id naming.
        {"name": "openai.gpt-oss-120b", "modelFormat": "Amazon", "modelVersion": "1", "sku": "OnDemand", "capacity": 1, "retirementDate": "2099-12-30"},
        {"name": "openai.gpt-oss-20b", "modelFormat": "Amazon", "modelVersion": "1", "sku": "OnDemand", "capacity": 1, "retirementDate": "2099-12-30"},
    ],
})

# ----- 5. GCP Gemini OpenAI-compat (gemini-openai) — fans into /v1/chat/completions ----
# Gemini's OpenAI-compat endpoint uses `Authorization: Bearer <key>`. Because the native
# Gemini backend (above) uses `x-goog-api-key: <raw key>` and APIM cannot prepend a
# `Bearer ` prefix at backend-resource level, this MUST point at a DIFFERENT Key Vault
# secret whose value is `Bearer <key>` (full Authorization header value).
extended_provider_backends.append({
    "backendId": "gemini-openai-compat-extended",
    "backendType": "gemini-openai",
    "authType": "api-key-bearer",
    "endpoint": "https://generativelanguage.googleapis.com",
    "priority": 1,
    "weight": 100,
    "authConfig": _auth_config(NV_GEMINI_OPENAI, gemini_openai_key_vault_uri, gemini_openai_inline),
    "supportedModels": [
        {"name": "gemini-2.5-flash-lite", "modelFormat": "Google", "modelVersion": "1", "sku": "Standard", "capacity": 1, "retirementDate": "2099-12-30"}
    ],
})

# ============================================================================
# OPTIONAL: existing Azure / Microsoft Foundry backends loaded from azd
# ============================================================================
existing_azure_backends: list = []          # populated below when init_from_azd=True

llm_backends_config = existing_azure_backends + extended_provider_backends

# ============================================================================
# 🪄 MODEL ALIASES — built further down (after the azd merge) using a scenario
# ----------------------------------------------------------------------------
# This phase of the accelerator does NOT translate between API protocols. Every
# alias must therefore front backends that share the SAME wire-level API spec
# (same path, same request/response shape, same auth contract). Three reference
# scenarios are emitted later in this cell — each one self-skips when the
# configured backends can't satisfy it. See the builder block at the bottom.
# ============================================================================
model_aliases: list = []          # populated after azd-loaded backends are merged

# Managed Identity for APIM authentication (auto-discovered if empty)
apim_managed_identity_name = ""

# ============================================================================
# 🔁 azd ENVIRONMENT OVERRIDES (only when init_from_azd = True)
# ============================================================================
if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group": ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":       ["AZURE_LOCATION", "LOCATION"],
        "llm_backends":   (["LLM_BACKEND_CONFIG", "LLM_BACKENDS_CONFIG"], "json"),
        "key_vault_name": ["KEY_VAULT_NAME"],
    }, verbose=False)

    if "resource_group" in loaded:  governance_hub_resource_group = loaded["resource_group"]
    if "location" in loaded:        location = loaded["location"]
    if "llm_backends" in loaded and isinstance(loaded["llm_backends"], list):
        llm_backends_config = loaded["llm_backends"] + extended_provider_backends
    if "key_vault_name" in loaded and not key_vault_name:
        key_vault_name = loaded["key_vault_name"]

    utils.print_ok(f"Resource group : {governance_hub_resource_group}")
    utils.print_ok(f"Location       : {location}")
    utils.print_ok(f"Backends loaded: {len(llm_backends_config)} (incl. {len(extended_provider_backends)} extended providers)")
    if key_vault_name:
        utils.print_ok(f"Key Vault      : {key_vault_name}")

# Track which backends have real credentials so test cells can skip cleanly. We
# check per-backend (not per-provider) because native and OpenAI-compat surfaces
# use independent secrets that may be configured separately.
def _backend_enabled(kv_uri: str, inline: str) -> bool:
    return bool(kv_uri) or not _is_placeholder(inline)

provider_enabled = {
    "aws_bedrock_native": _backend_enabled(aws_bedrock_native_key_vault_uri, aws_bedrock_native_inline),
    "aws_bedrock_openai": _backend_enabled(aws_bedrock_openai_key_vault_uri, aws_bedrock_openai_inline),
    "gemini_native":      _backend_enabled(gemini_native_key_vault_uri,      gemini_native_inline),
    "gemini_openai":      _backend_enabled(gemini_openai_key_vault_uri,      gemini_openai_inline),
    "anthropic":          _backend_enabled(anthropic_key_vault_uri,          anthropic_inline),
}
# Backwards-compat aggregate keys used by existing test cells (true if any surface
# of that provider is enabled).
provider_enabled["aws_bedrock"] = provider_enabled["aws_bedrock_native"] or provider_enabled["aws_bedrock_openai"]
provider_enabled["gemini"]      = provider_enabled["gemini_native"]      or provider_enabled["gemini_openai"]

utils.print_info(f"Initialization completed.")
utils.print_info(f"Total backends configured: {len(llm_backends_config)}")
utils.print_info(f"Backends enabled         : {[k for k, v in provider_enabled.items() if v] or 'NONE — fill REPLACE_* values to enable tests'}")

# ============================================================================
# 🪄 BUILD MODEL ALIASES from scenario templates (each is opt-in)
# ----------------------------------------------------------------------------
# The accelerator does not currently translate between API protocols, so every
# alias must front backends that share the SAME wire-level API spec. We
# materialise three scenario templates here — each one self-skips when the
# configured `llm_backends_config` (now finalised after azd merge) cannot
# satisfy it. Edit the `picks` lists below to lock specific real model names if
# you want deterministic behaviour across runs.
# ============================================================================

def _models_of_backend_type(backend_type: str) -> list[str]:
    """Distinct chat-completion model names from every configured backend of the given type.

    Embedding models are excluded — they live under a different operation
    contract (`/embeddings` rather than `/chat/completions` / `/v1/messages`)
    and grouping them with chat-completion models in the same alias would
    break the same-API-spec discipline."""
    out: list[str] = []
    for backend in llm_backends_config:
        if backend.get("backendType") != backend_type:
            continue
        for m in backend.get("supportedModels", []):
            name = m["name"] if isinstance(m, dict) else m
            if not name or "embedding" in name.lower():
                continue
            if name not in out:
                out.append(name)
    return out

_foundry_models        = _models_of_backend_type("ai-foundry")
_bedrock_mantle_models = _models_of_backend_type("aws-bedrock-mantle")
_gemini_openai_models  = _models_of_backend_type("gemini-openai")
_anthropic_models      = _models_of_backend_type("anthropic")

# --- Scenario A: Foundry weighted load-balance (single cloud, OpenAI-compat) ---
# Route between multiple Microsoft Foundry models behind one client-facing name —
# e.g. mix gpt-5 / Mistral / Phi-4 traffic with explicit weights. The alias only
# resolves to OpenAI-compat-capable members, so it works on the Universal LLM API
# and Unified AI `/v1/chat/completions`. Test surfaces below pick the same.
if len(_foundry_models) >= 2:
    _foundry_picks   = _foundry_models[:3]
    _foundry_weights = [50, 30, 20][: len(_foundry_picks)] if len(_foundry_picks) >= 3 else [60, 40]
    model_aliases.append({
        "name":     "foundry-weighted-mix",
        "models":   _foundry_picks,
        "strategy": "weighted",
        "weights":  _foundry_weights,
        # Notebook-only metadata (ignored by Bicep formatter):
        "_test_surfaces": ["universal-llm", "unified-openai-compat"],
        "_description":   "Microsoft Foundry weighted mix — single cloud, OpenAI-compat spec",
    })

# --- Scenario B: Cross-cloud OpenAI-compat alias ---
# Same `/v1/chat/completions` (OpenAI) spec across three clouds. Each member must
# be OpenAI-compat-capable: ai-foundry / aws-bedrock-mantle / gemini-openai.
# Priority strategy gives a primary + transparent cross-cloud fallback.
_multi_cloud_openai_picks: list[str] = []
if _foundry_models:        _multi_cloud_openai_picks.append(_foundry_models[0])
if _bedrock_mantle_models: _multi_cloud_openai_picks.append(_bedrock_mantle_models[0])
if _gemini_openai_models:  _multi_cloud_openai_picks.append(_gemini_openai_models[0])
if len(_multi_cloud_openai_picks) >= 2:
    model_aliases.append({
        "name":     "multi-cloud-openai",
        "models":   _multi_cloud_openai_picks,
        "strategy": "priority",
        "_test_surfaces": ["universal-llm", "unified-openai-compat"],
        "_description":   "Foundry + Bedrock-Mantle + Gemini-OpenAI — same OpenAI `/v1/chat/completions` spec across clouds",
    })

# --- Scenario C: Native Anthropic Messages alias (`/claude/v1/messages`) ---
# Same Anthropic Messages spec across providers. TODAY the only backend type that
# natively serves `/v1/messages` is `anthropic`, so the alias members are limited
# to direct Anthropic API keys (multi-region, multi-tenant, etc.). FUTURE work
# (tracked outside this notebook) will introduce two more `claude-native`-pool
# backend types so the same alias also fans into AWS Bedrock and Microsoft Foundry
# with no caller-side change:
#   - `aws-bedrock-anthropic-passthrough` (Bedrock's `anthropic.claude-*` Messages endpoint)
#   - `foundry-anthropic-passthrough`     (Foundry catalog Claude served via /v1/messages)
# The alias definition below is forward-compatible — once those backend types ship,
# add their model names to `_anthropic_models` and they slot in transparently.
if _anthropic_models:
    model_aliases.append({
        "name":     "multi-cloud-claude",
        "models":   _anthropic_models[:3],
        "strategy": "priority",
        "_test_surfaces": ["unified-claude-native"],
        "_description":   "Anthropic Messages `/v1/messages` spec — Anthropic backends today; Bedrock/Foundry passthroughs are future work",
    })

if model_aliases:
    utils.print_ok(f"Model aliases built : {len(model_aliases)}")
    for _a in model_aliases:
        _underlying = ", ".join(_a["models"])
        _strategy   = _a.get("strategy", "priority")
        _w          = f" weights={_a['weights']}" if _a.get("weights") else ""
        print(f"  • {_a['name']}  (strategy: {_strategy}{_w})")
        print(f"      {_a.get('_description', '')}")
        print(f"      members: [{_underlying}]")
else:
    utils.print_warning("No model aliases built — none of the 3 scenarios had enough members in llm_backends_config to be useful.")

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected. Initialize the APIM client to interact with your deployment:

In [ ]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()
    
    apim_resource_name = apimClientTool.apim_resource_name
    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    
    utils.print_ok(f"APIM Client Tool initialized successfully!")
    utils.print_info(f"APIM Resource Name: {apim_resource_name}")
    utils.print_info(f"APIM Gateway URL: {apim_resource_gateway_url}")
    
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

<a id='3'></a>
### 3️⃣ Extract Current APIM Backend-Pools Configuration

Retrieve and analyze the existing backend pools and backends configured in your APIM instance:

In [ ]:
# Extract current backends from APIM using the SDK
utils.print_info("Extracting current APIM backends configuration...")

try:
    # Use the APIMClientTool's new get_backends method (uses Azure SDK instead of CLI)
    existing_backends, existing_backend_pools = apimClientTool.get_backends()
    
except Exception as e:
    utils.print_error(f"Error extracting backends: {e}")
    existing_backends = []
    existing_backend_pools = []

In [ ]:
# Get supported models from the policy fragment (if exists)
try:
    supported_models_from_policy = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Supported models in APIM policy fragment 'set-backend-pools':")
    for model in supported_models_from_policy:
        print(f"  • {model}")
except Exception as e:
    utils.print_warning(f"Could not retrieve policy fragment (may not exist yet): {e}")
    supported_models_from_policy = []

In [ ]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

<a id='4'></a>
### 4️⃣ Discover Managed Identity for APIM Authentication

Auto-discover or specify the user-assigned managed identity used by APIM:

In [ ]:
# Discover managed identity from APIM using the SDK
utils.print_info("Discovering managed identity configuration...")

# Use the APIMClientTool's get_managed_identity_info method
managed_identity_info = apimClientTool.get_managed_identity_info()

managed_identity_client_id = managed_identity_info.get('clientId')
managed_identity_name = managed_identity_info.get('name') or apim_managed_identity_name
managed_identity_resource_group = managed_identity_info.get('resourceGroup') or governance_hub_resource_group

if not managed_identity_client_id:
    utils.print_warning("Could not auto-discover managed identity. Please specify it manually in the configuration.")
else:
    utils.print_info(f"Client ID: {managed_identity_client_id}")

if managed_identity_name:
    utils.print_ok(f"Managed Identity Name: {managed_identity_name}")
    utils.print_ok(f"Managed Identity Resource Group: {managed_identity_resource_group}")

<a id='5'></a>
### 5️⃣ Generate LLM Backend Parameter File

Generate a customizable `.bicepparam` file with the full list of LLM backends to be integrated with APIM:

In [ ]:
# Configure the LLM backends for deployment
# You can modify the llm_backends_config list defined in the initialization cell

utils.print_info("LLM Backends to be deployed:")
for backend in llm_backends_config:
    print(f"\n  🔗 {backend['backendId']}")
    print(f"     Type: {backend['backendType']}")
    print(f"     Endpoint: {backend['endpoint']}")
    auth = backend.get('authType') or backend.get('authScheme') or 'unspecified'
    print(f"     Auth: {auth}")
    auth_config = backend.get('authConfig')
    if auth_config:
        nv = auth_config.get('namedValueKey', 'N/A')
        if auth_config.get('keyVaultSecretUri'):
            print(f"     Auth Source: Key Vault → {auth_config['keyVaultSecretUri']} (named value: {nv})")
        elif auth_config.get('secretValue'):
            print(f"     Auth Source: inline secret (named value: {nv}) ⚠️ testing only")
        else:
            print(f"     Auth Source: named value '{nv}'")
    print(f"     Priority: {backend.get('priority', 1)}, Weight: {backend.get('weight', 100)}")
    # Display supported models with their per-model metadata
    print(f"     Models ({len(backend['supportedModels'])}):")
    for model in backend['supportedModels']:
        model_name = model['name'] if isinstance(model, dict) else model
        if isinstance(model, dict):
            sku = model.get('sku', 'Standard')
            capacity = model.get('capacity', 100)
            fmt = model.get('modelFormat', 'OpenAI')
            ver = model.get('modelVersion', '1')
            extras = []
            if 'retirementDate' in model:
                extras.append(f"Retirement: {model['retirementDate']}")
            if 'apiVersion' in model:
                extras.append(f"API: {model['apiVersion']}")
            if 'timeout' in model:
                extras.append(f"Timeout: {model['timeout']}s")
            if 'inferenceApiVersion' in model:
                extras.append(f"Inference API: {model['inferenceApiVersion']}")
            extra_str = f", {', '.join(extras)}" if extras else ""
            print(f"       - {model_name} (SKU: {sku}, Capacity: {capacity}, Format: {fmt}, Version: {ver}{extra_str})")
        else:
            print(f"       - {model_name}")

# Display model aliases if defined
if model_aliases:
    utils.print_info(f"\nModel Aliases ({len(model_aliases)}):")
    for alias in model_aliases:
        strategy = alias.get('strategy', 'priority')
        models_str = ', '.join(alias.get('models', []))
        weights = alias.get('weights')
        weights_str = f" weights={weights}" if weights else ""
        print(f"  • {alias['name']} → [{models_str}]  (strategy: {strategy}{weights_str})")


In [ ]:
# Generate the .bicepparam file content
bicep_dir = "../bicep/infra/llm-backend-onboarding"
params_file = os.path.join(bicep_dir, "llm-backends-generated-local.bicepparam")

# Format a single model object for Bicep
def format_model_for_bicep(model):
    """Format a model object for Bicep with per-model metadata including optional routing attributes."""
    if isinstance(model, str):
        # Legacy format: just model name string - convert to object with defaults
        return f"{{ name: '{model}' }}"
    
    # New format: model object with metadata
    parts = [f"name: '{model['name']}'"]
    if 'sku' in model:
        parts.append(f"sku: '{model['sku']}'")
    if 'capacity' in model:
        parts.append(f"capacity: {model['capacity']}")
    if 'modelFormat' in model:
        parts.append(f"modelFormat: '{model['modelFormat']}'")
    if 'modelVersion' in model:
        parts.append(f"modelVersion: '{model['modelVersion']}'")
    if 'retirementDate' in model:
        parts.append(f"retirementDate: '{model['retirementDate']}'")
    if 'apiVersion' in model:
        parts.append(f"apiVersion: '{model['apiVersion']}'")
    if 'timeout' in model:
        parts.append(f"timeout: {model['timeout']}")
    if 'inferenceApiVersion' in model:
        parts.append(f"inferenceApiVersion: '{model['inferenceApiVersion']}'")
    
    return "{ " + ", ".join(parts) + " }"

# Format an authConfig object for Bicep
def format_auth_config_for_bicep(auth_config):
    parts = []
    if 'namedValueKey' in auth_config:
        parts.append(f"namedValueKey: '{auth_config['namedValueKey']}'")
    if 'keyVaultSecretUri' in auth_config:
        parts.append(f"keyVaultSecretUri: '{auth_config['keyVaultSecretUri']}'")
    if 'secretValue' in auth_config:
        parts.append(f"secretValue: '{auth_config['secretValue']}'")
    return "{ " + ", ".join(parts) + " }"

# Format backends array for Bicep (uses per-model metadata)
def format_backend_for_bicep(backend):
    """Format a backend configuration for Bicep with per-model metadata."""
    # Format each model with its individual metadata
    models_formatted = [format_model_for_bicep(m) for m in backend['supportedModels']]
    models_str = "\n      ".join(models_formatted)

    # Auth: prefer new authType, fall back to legacy authScheme
    auth_lines = []
    if 'authType' in backend:
        auth_lines.append(f"    authType: '{backend['authType']}'")
    elif 'authScheme' in backend:
        auth_lines.append(f"    authScheme: '{backend['authScheme']}'")
    if 'authConfig' in backend:
        auth_lines.append(f"    authConfig: {format_auth_config_for_bicep(backend['authConfig'])}")
    auth_block = ("\n".join(auth_lines) + "\n") if auth_lines else ""

    return f"""  {{
    backendId: '{backend['backendId']}'
    backendType: '{backend['backendType']}'
    endpoint: '{backend['endpoint']}'
{auth_block}    supportedModels: [
      {models_str}
    ]
    priority: {backend.get('priority', 1)}
    weight: {backend.get('weight', 100)}
  }}"""

# Format model alias array for Bicep
def format_alias_for_bicep(alias):
    parts = [f"name: '{alias['name']}'"]
    models_arr = ", ".join([f"'{m}'" for m in alias.get('models', [])])
    parts.append(f"models: [ {models_arr} ]")
    if 'strategy' in alias:
        parts.append(f"strategy: '{alias['strategy']}'")
    if 'weights' in alias:
        weights_arr = ", ".join([str(w) for w in alias['weights']])
        parts.append(f"weights: [ {weights_arr} ]")
    return "  { " + ", ".join(parts) + " }"

backends_bicep_str = "\n".join([format_backend_for_bicep(b) for b in llm_backends_config])

# Optional sections
aliases_block = ""
if model_aliases:
    aliases_str = "\n".join([format_alias_for_bicep(a) for a in model_aliases])
    aliases_block = f"""

// ============================================================================
// Model Aliases group multiple models under a single client-facing name
// ============================================================================
param modelAliases = [
{aliases_str}
]
"""
else:
    aliases_block = "\n\nparam modelAliases = []\n"

key_vault_block = ""
if key_vault_name:
    key_vault_block = f"""
// ============================================================================
// Key Vault for backend credential references (used by authConfig.keyVaultSecretUri)
// ============================================================================
param keyVaultName = '{key_vault_name}'
"""

aws_block = ""
if True:  # always emit so anthropicVersion is set
    aws_block = f"""
// ============================================================================
// AWS credentials for Amazon Bedrock backends (stored as APIM secret named values)
// ============================================================================
// AWS SigV4 not used in this notebook — Bedrock backend uses api-key-bearer.
// The aws-access-key / aws-secret-key named values are still created with
// NOT_CONFIGURED defaults so the policy fragment compiles.
param awsRegion = '{aws_bedrock_region}'
param anthropicVersion = '{anthropic_version}'
"""

params_content = f"""using './main.bicep'

// ============================================================================
// LLM Backend Onboarding - Generated Parameter File
// Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}
// ============================================================================

// ============================================================================
// API Management (APIM) Configuration
// ============================================================================
param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apim_resource_name}'
}}

// ============================================================================
// APIM Managed Identity Configuration
// ============================================================================
param apimManagedIdentity = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{managed_identity_resource_group}'
  name: '{managed_identity_name}'
}}

// ============================================================================
// LLM Backend Configuration Array
// Each backend uses 'authType' (managed-identity | aws-sigv4 | api-key-bearer
//   | api-key-header | none); api-key auth types also require 'authConfig'.
// Each model in supportedModels has its own metadata (sku, capacity,
//   modelFormat, modelVersion, retirementDate, apiVersion, timeout,
//   inferenceApiVersion).
// ============================================================================
param llmBackendConfig = [
{backends_bicep_str}
]

// ============================================================================
// Circuit Breaker Configuration
// ============================================================================
param configureCircuitBreaker = true
{aliases_block}{key_vault_block}{aws_block}"""

# Write the parameter file
utils.print_info(f"Generating parameter file: {params_file}")
with open(params_file, 'w') as f:
    f.write(params_content)

utils.print_ok(f"Parameter file generated successfully!")
print("\n" + "="*60)
print("GENERATED PARAMETER FILE CONTENT:")
print("="*60)
print(params_content)


<a id='6'></a>
### 6️⃣ Deploy LLM Backend Onboarding Bicep

Deploy the LLM backends, backend pools, and policy fragments to APIM:

In [ ]:
# Deploy the LLM backend onboarding
deployment_name = f"llm-backend-onboarding-{time.strftime('%Y%m%d%H%M%S')}"
template_file = os.path.join(bicep_dir, "main.bicep")

utils.print_info(f"Starting deployment: {deployment_name}")
utils.print_info(f"Template: {template_file}")
utils.print_info(f"Parameters: {params_file}")

# Run the subscription-level deployment
deployment_cmd = f"az deployment sub create --name {deployment_name} --location {location} --template-file {template_file} --parameters {params_file}"

output = utils.run(
    deployment_cmd,
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

if output.success:
    utils.print_ok("Deployment completed successfully!")
    
    # Display deployment outputs if available
    outputs = output.json_data.get('properties', {}).get('outputs', {}) if output.json_data else {}
    
    if outputs:
        print("\n" + "="*60)
        print("DEPLOYMENT OUTPUTS:")
        print("="*60)
        
        for key, value in outputs.items():
            print(f"  {key}: {value.get('value')}")
    else:
        utils.print_info("No deployment outputs returned.")
else:
    utils.print_error("Deployment failed. Check the error messages above.")

<a id='7'></a>
### 7️⃣ Verify Deployed Configuration

Verify that the backends, pools, and policy fragments were created successfully:

In [ ]:
# Re-initialize APIM client to pick up new backends
apimClientTool.initialize()

# Get updated supported models from policy fragment
try:
    updated_supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Updated supported models in APIM policy fragment 'set-backend-pools':")
    for model in updated_supported_models:
        print(f"  • {model}")
except Exception as e:
    utils.print_error(f"Error retrieving policy fragment: {e}")
    updated_supported_models = []

In [ ]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

<a id='8'></a>
### 8️⃣ Verify Get Available Models Policy Fragment

Verify that the new `get-available-models` policy fragment was created successfully:

In [ ]:
# Verify the get-available-models policy fragment exists
try:
    # Use the Azure SDK to check if the policy fragment exists
    policy_fragment = apimClientTool.client.policy_fragment.get(
        resource_group_name=apimClientTool.resource_group_name,
        service_name=apimClientTool.apim_resource_name,
        id="get-available-models"
    )
    
    if policy_fragment:
        utils.print_ok("Policy fragment 'get-available-models' exists!")
        utils.print_info("This fragment returns available model deployments in a format similar to Azure Cognitive Services API.")
        utils.print_info(f"Description: {policy_fragment.description}")
except Exception as e:
    utils.print_warning(f"Could not retrieve 'get-available-models' policy fragment: {e}")
    utils.print_info("This fragment will be created after running the deployment.")

---
## 9️⃣ Provision Access Contract

The validation tests below need an APIM subscription whose product policy allows
**every** model discovered during onboarding (Foundry inference, Bedrock-Mantle,
Gemini-OpenAI, native Bedrock, native Gemini, native Anthropic, plus any model
aliases). This step:

1. Builds `allowed_models_csv` from the union of:
   - `supportedModels[].name` across every backend in `llm_backends_config`
   - `model_aliases[].name`
2. Generates a contract folder under `bicep/infra/citadel-access-contracts/contracts/<bu-usecase>/<env>/`
   with a custom `ai-product-policy.xml` (sets `allowedModels` = the CSV) and a
   `main.bicepparam`.
3. Deploys the contract via `az deployment sub create`. The deployment creates an
   APIM **Product** `LLM-<BU>-<UseCase>-<ENV>` with subscription `…-SUB-01`.
4. Stores `product_id` and `subscription_name` for the discovery cell to consume.

The cell is **idempotent** — re-running with the same `(business_unit, use_case_name, environment)`
triple updates the existing product policy and reuses the same subscription key.


In [ ]:
# ============================================================================
# 9️⃣ Provision Access Contract — allows every onboarded model
# ============================================================================

# --- Contract identity (override here if you want a different product name) ---
contract_business_unit = "Testing"
contract_use_case_name = "ExtendedProviders"
contract_environment   = "DEV"

# --- 1. Build allowed_models_list from llm_backends_config + model_aliases ---
allowed_models_list: list[str] = []
seen_models: set[str] = set()

def _add_model(name):
    if name and name not in seen_models:
        seen_models.add(name)
        allowed_models_list.append(name)

for backend in llm_backends_config:
    for m in backend.get("supportedModels", []):
        _add_model(m["name"] if isinstance(m, dict) else m)

for alias in model_aliases or []:
    _add_model(alias["name"] if isinstance(alias, dict) else alias)

if not allowed_models_list:
    raise RuntimeError(
        "No models discovered from llm_backends_config or model_aliases — "
        "run the deploy + discovery cells first so the lists are populated."
    )

allowed_models_csv = ",".join(allowed_models_list)
utils.print_info(f"Allowed models for access contract ({len(allowed_models_list)}):")
for m in allowed_models_list:
    utils.print_info(f"  • {m}")

# --- 2. Materialize the contract folder, policy XML, and bicepparam ---
bicep_dir     = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

folder_name        = f"{contract_business_unit.lower()}-{contract_use_case_name.lower()}"
environment_folder = contract_environment.lower()
contract_folder    = os.path.join(bicep_dir, "contracts", folder_name, environment_folder)
os.makedirs(contract_folder, exist_ok=True)
utils.print_info(f"📁 Contract folder: {contract_folder}")

product_policy = f'''<policies>
    <inbound>
        <base />
        <!-- Extract and validate model parameter from request -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Allow ALL models discovered by the extended-providers onboarding run -->
        <set-variable name="allowedModels" value="{allowed_models_csv}" />

        <!-- Capacity management -->
        <llm-token-limit counter-key="@(context.Subscription.Id)" tokens-per-minute="1000" estimate-prompt-tokens="false" token-quota="100000" token-quota-period="Monthly" />

        <!-- Enable advanced response headers offered by set-response-headers fragment -->
        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>'''

policy_file_dest = os.path.join(contract_folder, "ai-product-policy.xml")
with open(policy_file_dest, "w") as f:
    f.write(product_policy)
utils.print_ok(f"📋 Custom product policy written: {policy_file_dest}")

params_file = os.path.join(contract_folder, "main.bicepparam")
params_content = f"""using '../../../main.bicep'

// ============================================================================
// Extended Providers Test Contract — generated from validation notebook
// Allows every model onboarded by this notebook (Foundry, Bedrock-Mantle,
// Gemini-OpenAI, native Bedrock, native Gemini, native Anthropic + aliases).
// ============================================================================

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  name: 'placeholder'
}}

param useTargetAzureKeyVault = false

param useCase = {{
  businessUnit: '{contract_business_unit}'
  useCaseName: '{contract_use_case_name}'
  environment: '{contract_environment}'
}}

param apiNameMapping = {{
  LLM: ['universal-llm-api', 'azure-openai-api', 'unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: 'EXTENDED-PROVIDERS-ENDPOINT'
    apiKeySecretName: 'EXTENDED-PROVIDERS-KEY'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = 'Extended providers test contract — generated from validation notebook'

// Azure AI Foundry Integration (disabled — onboarding doesn't push secrets to Foundry)
param useTargetFoundry = false

param foundry = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}}
"""

with open(params_file, "w") as f:
    f.write(params_content)
utils.print_ok(f"✅ Bicep param file written: {params_file}")

# --- 3. Deploy the contract ---
contract_name = f"extended-providers-contract-{time.strftime('%Y%m%d%H%M%S')}"
product_id        = f"LLM-{contract_business_unit}-{contract_use_case_name}-{contract_environment}"
subscription_name = f"{product_id}-SUB-01"

utils.print_info(f"Deploying access contract: {contract_name}")
utils.print_info(f"  Product ID       : {product_id}")
utils.print_info(f"  Subscription name: {subscription_name}")

deployment_cmd = (
    f"az deployment sub create "
    f"--name {contract_name} "
    f"--location {location} "
    f"--template-file {template_file} "
    f"--parameters {params_file}"
)
output = utils.run(
    deployment_cmd,
    f"Deployment '{contract_name}' succeeded",
    f"Deployment '{contract_name}' failed",
)

if output.success:
    utils.print_ok(f"✅ Access contract deployed — Product: {product_id}, Subscription: {subscription_name}")
else:
    utils.print_error("❌ Access contract deployment failed — fix the error above before running the validation tests.")

# --- 4. Refresh APIM client so the new subscription appears in apim_subscriptions ---
apimClientTool.initialize()
utils.print_ok("APIM client re-initialized — subscription list refreshed.")


---
## 🧪 Validation Tests

All test URLs are built from `{apim_gateway_url}` plus the API URL suffix configured in
APIM (`models` for Universal LLM, `unified-ai` for Unified AI), then the operation/path:

| API | Inbound path | Backend types tested | Surface |
|---|---|---|---|
| Universal LLM | `{gateway}/models/chat/completions` | `gemini-openai`, `aws-bedrock-mantle`, `ai-foundry` | OpenAI-compat |
| Unified AI | `{gateway}/unified-ai/v1/chat/completions` | `gemini-openai`, `aws-bedrock-mantle`, `ai-foundry` | OpenAI-compat |
| Unified AI | `{gateway}/unified-ai/bedrock/model/{model}/converse` | `aws-bedrock` | **Native** Bedrock Converse |
| Unified AI | `{gateway}/unified-ai/gemini/v1beta/models/{model}:generateContent` | `gemini` | **Native** Gemini |
| Unified AI | `{gateway}/unified-ai/claude/v1/messages` | `anthropic` | **Native** Anthropic Messages |

Each test cell **skips automatically** when the underlying provider's credential is still
`REPLACE_*` (see `provider_enabled` dict from cell 3). Test model lists are derived
automatically from the backends declared in `extended_provider_backends` (cell 3) so
edits made there are honoured by every test below.


<a id='test-helpers'></a>
### 🛠️ Shared helpers

In [ ]:
def is_unsupported_operation_400(response):
    """Return True for backend 400s that just mean the model doesn't support this surface."""
    if response.status_code != 400:
        return False
    text = (response.text or "").lower()
    markers = (
        "operation is unsupported",
        "operation_not_supported",
        "operationnotsupported",
        "unsupportedoperation",
        "not supported with the model",
    )
    return any(m in text for m in markers)


def print_uaig_debug(response):
    """Print Unified AI debug headers when present."""
    debug_keys = (
        "UAIG-Backend",
        "UAIG-API-Type",
        "UAIG-Final-Path",
        "UAIG-Model-Id",
        "x-ms-region",
    )
    seen = False
    for k in debug_keys:
        v = response.headers.get(k)
        if v:
            if not seen:
                print("    Debug headers:")
                seen = True
            print(f"      {k}: {v}")


def short_response(text, limit=400):
    return text if len(text) <= limit else text[:limit] + " ...[truncated]"


def skip_if_disabled(provider_key: str, label: str) -> bool:
    """Return True (and print a warning) when the provider's credentials are REPLACE_*."""
    if not provider_enabled.get(provider_key, False):
        utils.print_warning(f"{label} skipped — {provider_key} credentials are still REPLACE_*. Fill them in cell 3 to enable.")
        return True
    return False


<a id='test-discovery'></a>
### 🔎 Discover All Available Models (Unified AI)

Lists every model exposed by the gateway, including the new providers.

In [ ]:
# Resolve the API key — prefer the access-contract subscription created in the
# previous step (`subscription_name`), otherwise fall back to the first APIM
# subscription. The contract's product policy is what actually sets `allowedModels`,
# enables `enableResponseHeaders`, and applies the token-limit; tests will fail
# (or behave inconsistently) if a generic admin/master sub is used instead.
api_key = None
selected_sub = None

target_sub_name = (globals().get("subscription_name") or "").lower()

if target_sub_name and apimClientTool.apim_subscriptions:
    for sub in apimClientTool.apim_subscriptions:
        if target_sub_name in (sub.get("name") or "").lower():
            api_key      = sub.get("key")
            selected_sub = sub.get("name")
            break

if api_key is None and apimClientTool.apim_subscriptions:
    selected_sub = apimClientTool.apim_subscriptions[0].get("name")
    api_key      = apimClientTool.apim_subscriptions[0].get("key")
    utils.print_warning(
        f"Access-contract subscription '{target_sub_name}' not found — falling back to '{selected_sub}'. "
        "Re-run the Provision Access Contract cell if you expected a contract-scoped key."
    )

if api_key is None:
    utils.print_error("No APIM subscriptions found. Provision an Access Contract before running tests.")
else:
    utils.print_ok(f"Using subscription: {selected_sub}")

# ----------------------------------------------------------------------------
# Discover the API base URLs.
#
# `apimClientTool.discover_api(filter)` finds the API whose `path` contains
# `filter` and sets `azure_endpoint` to `{gateway}/{api.path stripped of filter}`.
# Because each API path equals its filter exactly here, the resulting endpoint
# always lands at `{gateway}/`. We therefore re-add the canonical APIM API URL
# suffix (`models/`, `unified-ai/`, `openai/`) explicitly when building URLs:
#
#   Universal LLM API : {gateway}/models/<operation>           (no `/v1`)
#   Azure OpenAI API  : {gateway}/openai/deployments/{model}/<operation>?api-version=...
#   Unified AI API    : {gateway}/unified-ai/<base-path>/<operation>
#                       e.g. /unified-ai/v1/chat/completions
#                            /unified-ai/bedrock/model/{id}/converse
#                            /unified-ai/gemini/v1beta/models/{id}:generateContent
#                            /unified-ai/claude/v1/messages
#                            /unified-ai/deployments?api-version=...
# ----------------------------------------------------------------------------

UNIVERSAL_LLM_PATH = "models"
UNIFIED_AI_PATH    = "unified-ai"
AZURE_OPENAI_PATH  = "openai"

apim_gateway_url = str(apimClientTool.apim_resource_gateway_url).rstrip("/")

universal_llm_root = None
unified_ai_root    = None

# Discover Unified AI API
try:
    apimClientTool.discover_api(UNIFIED_AI_PATH)
    unified_ai_root = f"{apim_gateway_url}/{UNIFIED_AI_PATH}/"
    utils.print_ok(f"Unified AI API base: {unified_ai_root}")
except Exception as e:
    utils.print_warning(f"Unified AI API not found in APIM: {e}")

# Discover Universal LLM API
try:
    apimClientTool.discover_api(UNIVERSAL_LLM_PATH)
    universal_llm_root = f"{apim_gateway_url}/{UNIVERSAL_LLM_PATH}/"
    utils.print_ok(f"Universal LLM API base: {universal_llm_root}")
except Exception as e:
    utils.print_warning(f"Universal LLM API not found: {e}")

# ----------------------------------------------------------------------------
# Build per-provider model lists from the backends configured in cell 3 so
# tests always honour whatever you put in `extended_provider_backends` (and
# any extra backends merged from azd).
# ----------------------------------------------------------------------------
def models_for_backend_type(backend_type: str) -> list[str]:
    """Return all supportedModel names from configured backends of the given type."""
    out = []
    seen = set()
    for backend in llm_backends_config:
        if backend.get("backendType") != backend_type:
            continue
        for m in backend.get("supportedModels", []):
            name = m["name"] if isinstance(m, dict) else m
            if name and name not in seen:
                seen.add(name)
                out.append(name)
    return out

# Discover all OpenAI-compat-capable models (Foundry + Bedrock-Mantle + Gemini-OpenAI).
OPENAI_COMPAT_BACKEND_TYPES = ("ai-foundry", "aws-bedrock-mantle", "gemini-openai")
PROVIDER_LABELS = {
    "ai-foundry":          "Microsoft Foundry",
    "aws-bedrock-mantle":  "AWS Bedrock (Mantle / OpenAI-compat)",
    "gemini-openai":       "GCP Gemini (OpenAI-compat)",
}

openai_compat_test_models: list[tuple[str, str]] = []  # (model_name, provider_label)
for backend_type in OPENAI_COMPAT_BACKEND_TYPES:
    for name in models_for_backend_type(backend_type):
        openai_compat_test_models.append((name, PROVIDER_LABELS.get(backend_type, backend_type)))

bedrock_native_models   = models_for_backend_type("aws-bedrock")
gemini_native_models    = models_for_backend_type("gemini")
anthropic_native_models = models_for_backend_type("anthropic")

utils.print_info(f"\nResolved test model lists from backend config:")
utils.print_info(f"  OpenAI-compat ({len(openai_compat_test_models)}): {[m for m, _ in openai_compat_test_models]}")
utils.print_info(f"  Bedrock native ({len(bedrock_native_models)}): {bedrock_native_models}")
utils.print_info(f"  Gemini native  ({len(gemini_native_models)}): {gemini_native_models}")
utils.print_info(f"  Anthropic native ({len(anthropic_native_models)}): {anthropic_native_models}")

# ----------------------------------------------------------------------------
# Discover all gateway-exposed deployments via the Unified AI surface.
# Path: {gateway}/unified-ai/deployments?api-version=...
# ----------------------------------------------------------------------------
if api_key and unified_ai_root:
    deployments_url = f"{unified_ai_root}deployments?api-version={inference_api_version}"
    utils.print_info(f"\nGET {deployments_url}")
    try:
        r = requests.get(deployments_url, headers={"api-key": api_key}, timeout=30)
        utils.print_response_code(r)
        if r.status_code == 200:
            data = r.json()
            deployments = data.get("value", [])
            utils.print_ok(f"Found {len(deployments)} deployment(s)")
            for d in deployments:
                name   = d.get("name", "?")
                btype  = (d.get("properties", {}) or {}).get("model", {}).get("format", "?")
                ddtype = d.get("type", "?")
                print(f"  • {name:<55} format={btype:<12} type={ddtype}")
        else:
            utils.print_error(r.text[:500])
    except Exception as exc:
        utils.print_error(f"Discovery call failed: {exc}")


<a id='test-openai-compat-universal'></a>
### 🧪 OpenAI-compat across providers — **Universal LLM API**

Universal LLM only exposes the OpenAI-compat surface. Same `/models/chat/completions`
inbound path serves Bedrock (via Mantle backend type), Gemini (via gemini-openai backend
type), and any Microsoft Foundry models you've onboarded — the gateway picks the backend
by matching the body's `model` field against backend pools whose `poolType` is
OpenAI-compat-capable (`compatible-pool-types` filter on the `openai-compat` api-type).

> Universal LLM is **OpenAI-v1 shaped** — operations sit directly under the `/models/`
> URL suffix (no `/v1` segment). The model lists are derived automatically from the
> backends configured in cell 3 (only `aws-bedrock-mantle`, `gemini-openai`, and
> `ai-foundry` backend types contribute models here).


In [ ]:
# Universal LLM API — OpenAI-compat surface, all providers fan into the same path.
# Inbound URL: {gateway}/models/chat/completions   (NO `/v1` segment)
# Models are taken from `openai_compat_test_models` derived in the discovery cell from
# `extended_provider_backends` + any azd-merged backends in `llm_backends_config`.

if api_key and universal_llm_root and openai_compat_test_models:
    chat_url = f"{universal_llm_root}chat/completions"
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user",   "content": "Reply with the single word OK."},
    ]
    utils.print_info(f"POST {chat_url}\n")
    for model_name, provider_label in openai_compat_test_models:
        utils.print_info(f"[{provider_label}] model: {model_name}")
        try:
            r = requests.post(chat_url, headers={"api-key": api_key},
                              json={"model": model_name, "messages": test_messages},
                              timeout=60)
            utils.print_response_code(r)
            print_uaig_debug(r)
            if r.status_code == 200:
                data = r.json()
                answer = (data.get("choices", [{}])[0].get("message", {}) or {}).get("content", "")
                print(f"  💬 {short_response(answer.strip())}")
                utils.print_ok(f"{model_name} - SUCCESS\n")
            elif is_unsupported_operation_400(r):
                utils.print_ok(f"{model_name} - SUCCESS (routed; backend rejected on operation)\n")
            else:
                utils.print_error(f"{model_name} - FAILED ({r.status_code}): {short_response(r.text)}\n")
        except Exception as exc:
            utils.print_error(f"{model_name} - ERROR: {exc}\n")
elif not openai_compat_test_models:
    utils.print_warning("No OpenAI-compat-capable backends configured — fill in REPLACE_* keys in cell 3 or add ai-foundry / aws-bedrock-mantle / gemini-openai backends.")
else:
    utils.print_warning("Universal LLM API tests skipped — missing API key or endpoint.")


<a id='test-openai-compat-unified'></a>
### 🧪 OpenAI-compat across providers — **Unified AI API**

Unified AI exposes the same OpenAI-compat surface at `/unified-ai/v1/chat/completions`.
The `openai-compat` api-type uses `backend-path-templates[poolType][operation]` to rewrite
to each provider's OpenAI-compat path:

| Backend type | Backend path |
|---|---|
| `ai-foundry` | `/openai/deployments/{model}/chat/completions` |
| `aws-bedrock-mantle` | `/v1/chat/completions` |
| `gemini-openai` | `/v1beta/openai/chat/completions` |

In [ ]:
# Unified AI API — OpenAI-compat surface (api-type `openai-compat` base-path `/v1`).
# Inbound URL: {gateway}/unified-ai/v1/chat/completions
# Reuses the same model list as the Universal LLM test so you can compare both surfaces.

unified_openai_compat_models = list(openai_compat_test_models)

if api_key and unified_ai_root and unified_openai_compat_models:
    chat_url = f"{unified_ai_root}v1/chat/completions"
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user",   "content": "Reply with the single word OK."},
    ]
    utils.print_info(f"POST {chat_url}\n")
    for model_name, provider_label in unified_openai_compat_models:
        utils.print_info(f"[{provider_label}] model: {model_name}")
        try:
            r = requests.post(chat_url, headers={"api-key": api_key},
                              json={"model": model_name, "messages": test_messages},
                              timeout=60)
            utils.print_response_code(r)
            print_uaig_debug(r)
            if r.status_code == 200:
                data = r.json()
                answer = (data.get("choices", [{}])[0].get("message", {}) or {}).get("content", "")
                print(f"  💬 {short_response(answer.strip())}")
                utils.print_ok(f"{model_name} - SUCCESS\n")
            elif is_unsupported_operation_400(r):
                utils.print_ok(f"{model_name} - SUCCESS (routed; backend rejected on operation)\n")
            else:
                utils.print_error(f"{model_name} - FAILED ({r.status_code}): {r.text}\n")
        except Exception as exc:
            utils.print_error(f"{model_name} - ERROR: {exc}\n")
elif not unified_openai_compat_models:
    utils.print_warning("No OpenAI-compat-capable backends configured.")
else:
    utils.print_warning("Unified AI openai-compat tests skipped.")


<a id='test-bedrock-native'></a>
### 🧪 Native AWS Bedrock — **Unified AI `/bedrock/` prefix**

Inbound path: `/unified-ai/bedrock/model/{escapedModelId}/converse`

Routing flow:
1. `frag-request-processor.xml` matches the `bedrock-native` api-type by `/bedrock` base-path,
   sets `compatiblePoolTypes='aws-bedrock'`, and extracts the model id from the `/model/...`
   segment of the path.
2. `frag-set-target-backend-pool.xml` filters pools to those with `poolType='aws-bedrock'` and
   matches the model id against `supportedModels`.
3. `frag-path-builder.xml` strips the `/bedrock` prefix, leaving `/model/{escapedModelId}/converse`,
   and forwards the body unchanged.
4. `frag-set-backend-authorization.xml` applies `api-key-bearer` (`Authorization: Bearer <Bedrock API key>`).

Request/response use the **native** Bedrock Converse schema.

In [ ]:
# Native AWS Bedrock — Unified AI `/bedrock/` prefix.
# Inbound URL: {gateway}/unified-ai/bedrock/model/{escapedModelId}/converse
# Models come from `aws-bedrock` backends in `llm_backends_config` (see discovery cell).

if not skip_if_disabled("aws_bedrock", "Bedrock native test") and api_key and unified_ai_root:
    if not bedrock_native_models:
        utils.print_warning("No `aws-bedrock` backends configured — nothing to test.")
    for model_id in bedrock_native_models:
        encoded = quote(model_id, safe="")
        url = f"{unified_ai_root}bedrock/model/{encoded}/converse"
        utils.print_info(f"\nPOST {url}")
        body = {
            "messages": [{"role": "user", "content": [{"text": "Reply with the single word OK."}]}],
            "inferenceConfig": {"maxTokens": 32, "temperature": 0.0},
        }
        try:
            r = requests.post(url, headers={"api-key": api_key, "Content-Type": "application/json"},
                              json=body, timeout=60)
            utils.print_response_code(r)
            print_uaig_debug(r)
            if r.status_code == 200:
                data = r.json()
                output = (data.get("output", {}) or {}).get("message", {}) or {}
                blocks = output.get("content", [])
                text = "".join(b.get("text", "") for b in blocks if isinstance(b, dict))
                print(f"  💬 {short_response(text.strip())}")
                utils.print_ok(f"{model_id} - SUCCESS\n")
            else:
                utils.print_error(f"{model_id} - FAILED ({r.status_code}): {short_response(r.text)}\n")
        except Exception as exc:
            utils.print_error(f"{model_id} - ERROR: {exc}\n")


<a id='test-gemini-native'></a>
### 🧪 Native GCP Gemini — **Unified AI `/gemini/` prefix**

Inbound path: `/unified-ai/gemini/v1beta/models/{model}:generateContent`

Routing flow:
1. `frag-request-processor.xml` matches `gemini-native` api-type by `/gemini` base-path,
   sets `compatiblePoolTypes='gemini'`, and extracts the model id from the
   `/models/{model}:` segment.
2. Pool filter restricts selection to `gemini` poolType.
3. `frag-path-builder.xml` strips `/gemini`, leaving `/v1beta/models/{model}:generateContent`,
   and forwards the body unchanged.
4. Auth fragment sets `x-goog-api-key` from the `gemini-extended-api-key` named value
   (Key Vault reference or inline value).

In [ ]:
# Native GCP Gemini — Unified AI `/gemini/` prefix.
# Inbound URL: {gateway}/unified-ai/gemini/v1beta/models/{model}:generateContent
# Models come from `gemini` backends in `llm_backends_config` (see discovery cell).

if not skip_if_disabled("gemini", "Gemini native test") and api_key and unified_ai_root:
    if not gemini_native_models:
        utils.print_warning("No `gemini` backends configured — nothing to test.")
    for model_id in gemini_native_models:
        url = f"{unified_ai_root}gemini/v1beta/models/{model_id}:generateContent"
        utils.print_info(f"\nPOST {url}")
        # Native Gemini body schema — see https://ai.google.dev/api/generate-content
        body = {
            "contents": [{"role": "user", "parts": [{"text": "Reply with the single word OK."}]}],
            "generationConfig": {"maxOutputTokens": 32, "temperature": 0.0},
        }
        try:
            r = requests.post(url, headers={"api-key": api_key, "Content-Type": "application/json"},
                              json=body, timeout=60)
            utils.print_response_code(r)
            print_uaig_debug(r)
            if r.status_code == 200:
                data = r.json()
                cands = data.get("candidates", [])
                parts = (cands[0].get("content", {}) if cands else {}).get("parts", [])
                text = "".join(p.get("text", "") for p in parts if isinstance(p, dict))
                print(f"  💬 {short_response(text.strip())}")
                utils.print_ok(f"{model_id} - SUCCESS\n")
            else:
                utils.print_error(f"{model_id} - FAILED ({r.status_code}): {r.text}\n")
        except Exception as exc:
            utils.print_error(f"{model_id} - ERROR: {exc}\n")


<a id='test-anthropic-native'></a>
### 🧪 Native Anthropic Claude — **Unified AI `/claude/` prefix**

Inbound path: `/unified-ai/claude/v1/messages`

Routing flow:
1. `frag-request-processor.xml` matches `claude-native` api-type by `/claude` base-path,
   sets `compatiblePoolTypes='anthropic'`. Model id is extracted from the request body's
   `model` field (Anthropic Messages API has no model in the URL).
2. Pool filter restricts selection to `anthropic` poolType.
3. `frag-path-builder.xml` always sets the final path to `/v1/messages` and ensures the
   body's `model` field is populated.
4. Auth fragment sets `x-api-key` from `anthropic-extended-api-key` and `anthropic-version`
   from the `{{anthropic-version}}` named value (default `2023-06-01`).

Request/response use the **native** Anthropic Messages schema.

In [ ]:
# Native Anthropic Claude — Unified AI `/claude/` prefix.
# Inbound URL: {gateway}/unified-ai/claude/v1/messages
# Models come from `anthropic` backends in `llm_backends_config` (see discovery cell).

if not skip_if_disabled("anthropic", "Anthropic native test") and api_key and unified_ai_root:
    if not anthropic_native_models:
        utils.print_warning("No `anthropic` backends configured — nothing to test.")
    url = f"{unified_ai_root}claude/v1/messages"
    utils.print_info(f"POST {url}\n")
    for model_name in anthropic_native_models:
        utils.print_info(f"Model: {model_name}")
        body = {
            "model": model_name,
            "max_tokens": 32,
            "messages": [{"role": "user", "content": "Reply with the single word OK."}],
        }
        try:
            r = requests.post(url, headers={"api-key": api_key, "Content-Type": "application/json"},
                              json=body, timeout=60)
            utils.print_response_code(r)
            print_uaig_debug(r)
            if r.status_code == 200:
                data = r.json()
                content = data.get("content", [])
                text = "".join(b.get("text", "") for b in content if isinstance(b, dict))
                print(f"  💬 {short_response(text.strip())}")
                utils.print_ok(f"{model_name} - SUCCESS\n")
            else:
                utils.print_error(f"{model_name} - FAILED ({r.status_code}): {short_response(r.text)}\n")
        except Exception as exc:
            utils.print_error(f"{model_name} - ERROR: {exc}\n")


<a id='test-multi-cloud-alias'></a>
### 🧪 Model alias scenarios — same-API-spec routing

> **Phase scope.** This phase of the accelerator does **not** translate between API
> protocols. Every alias must therefore front backends that share the **same
> wire-level API spec** (same path + same request/response shape + same auth
> contract). Inbound requests are routed unchanged to the picked member's underlying
> pool — only the JSON body's `model` field is rewritten to the resolved real model
> name. Future work will add per-protocol passthrough backend types (e.g.
> `aws-bedrock-anthropic-passthrough`) so a single alias can fan across clouds even
> when each cloud exposes the same model under a different native API.

The init cell builds **up to three opt-in scenarios** from the configured backends —
each one materialises only when the underlying members exist:

| Scenario | Alias name | API spec | Members today | Test surface |
|---|---|---|---|---|
| **A. Foundry weighted** | `foundry-weighted-mix` | OpenAI `/v1/chat/completions` (single cloud) | Up to 3 Microsoft Foundry models, weighted | Universal LLM + Unified AI `/v1/chat/completions` |
| **B. Cross-cloud OpenAI-compat** | `multi-cloud-openai` | OpenAI `/v1/chat/completions` (multi-cloud) | First model from each of: Foundry, Bedrock-Mantle, Gemini-OpenAI | Universal LLM + Unified AI `/v1/chat/completions` |
| **C. Native Anthropic Messages** | `multi-cloud-claude` | Anthropic `/v1/messages` | Anthropic API-key members (today). Future: Bedrock-Anthropic + Foundry-Anthropic passthroughs | Unified AI `/unified-ai/claude/v1/messages` |

**What you should observe:**
- The `UAIG-Model-Id` response header reveals which underlying member the alias resolved to (e.g. for the weighted scenario, expect a roughly `50/30/20` split across runs).
- `UAIG-Backend` reveals the resolved backend pool name — confirming cross-cloud routing for scenario B.
- For scenario C, all calls today resolve to an `anthropic` member; once Bedrock/Foundry passthrough backend types ship, the same alias resolution will fan across providers with **no client-side change**.

If a scenario's alias was not built (insufficient backends), the test loop simply skips it.



In [ ]:
# ============================================================================
# Model alias scenario tests — each alias is exercised on the surfaces tagged
# in its `_test_surfaces` field by the init cell. Same-API-spec routing only:
# every alias member is guaranteed to live under the same wire-level contract,
# so the gateway hits the picked member's pool directly without any body or
# path translation (only the JSON body's `model` field is rewritten).
#
# Surfaces:
#   • universal-llm           → POST {gateway}/models/chat/completions
#   • unified-openai-compat   → POST {gateway}/unified-ai/v1/chat/completions
#   • unified-claude-native   → POST {gateway}/unified-ai/claude/v1/messages
# ============================================================================

if not model_aliases:
    utils.print_warning(
        "No model aliases configured — skipping alias scenario tests. The init cell builds "
        "scenarios A/B/C only when the configured backends can satisfy them."
    )
elif not api_key:
    utils.print_warning("Alias scenario tests skipped — no APIM subscription key was resolved.")
else:
    test_messages_openai = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user",   "content": "Reply with the single word OK."},
    ]

    def _claude_messages_body(alias_name: str) -> dict:
        # Anthropic Messages API takes the model in the body. Sending the alias here is fine —
        # set-target-backend-pool resolves it server-side and rewrites the body to the picked
        # member name before forwarding.
        return {
            "model": alias_name,
            "max_tokens": 32,
            "messages": [{"role": "user", "content": "Reply with the single word OK."}],
        }

    surface_resolvers = {
        "universal-llm":         lambda: (f"{universal_llm_root}chat/completions",     "Universal LLM API",            "openai")    if universal_llm_root else None,
        "unified-openai-compat": lambda: (f"{unified_ai_root}v1/chat/completions",     "Unified AI / OpenAI-compat",   "openai")    if unified_ai_root    else None,
        "unified-claude-native": lambda: (f"{unified_ai_root}claude/v1/messages",      "Unified AI / Claude native",   "anthropic") if unified_ai_root    else None,
    }

    for alias in model_aliases:
        alias_name      = alias["name"]
        underlying      = ", ".join(alias["models"])
        strategy        = alias.get("strategy", "priority")
        weights         = alias.get("weights")
        weights_str     = f" weights={weights}" if weights else ""
        description     = alias.get("_description", "")
        test_surfaces   = alias.get("_test_surfaces", [])

        utils.print_info(f"\n{'='*78}")
        utils.print_info(f"🔁 Alias: {alias_name}  (strategy: {strategy}{weights_str})")
        if description:
            utils.print_info(f"   {description}")
        utils.print_info(f"   underlying members: [{underlying}]")
        utils.print_info('='*78)

        for surface_key in test_surfaces:
            resolver = surface_resolvers.get(surface_key)
            resolved = resolver() if resolver else None
            if not resolved:
                utils.print_warning(f"   surface '{surface_key}' not available (API endpoint not discovered) — skipping")
                continue
            url, surface_label, body_shape = resolved

            if body_shape == "openai":
                body    = {"model": alias_name, "messages": test_messages_openai, "max_tokens": 32}
                headers = {"api-key": api_key}
            else:  # anthropic
                body    = _claude_messages_body(alias_name)
                headers = {"api-key": api_key, "Content-Type": "application/json"}

            utils.print_info(f"   via {surface_label} → POST {url}")
            try:
                r = requests.post(url, headers=headers, json=body, timeout=60)
                utils.print_response_code(r)
                print_uaig_debug(r)
                if r.status_code == 200:
                    if body_shape == "openai":
                        data   = r.json()
                        answer = (data.get("choices", [{}])[0].get("message", {}) or {}).get("content", "")
                    else:
                        data   = r.json()
                        content = data.get("content", [])
                        answer  = "".join(b.get("text", "") for b in content if isinstance(b, dict))
                    print(f"     💬 {short_response(answer.strip())}")
                    utils.print_ok(f"{alias_name} via {surface_label} - SUCCESS\n")
                elif r.status_code == 400 and "alias_no_compatible_member" in (r.text or "").lower():
                    utils.print_warning(
                        f"{alias_name} via {surface_label} - alias has no member compatible with this surface. "
                        "Add a backend member whose pool type matches the surface, or test on a different surface.\n"
                    )
                else:
                    utils.print_error(f"{alias_name} via {surface_label} - FAILED ({r.status_code}): {short_response(r.text)}\n")
            except Exception as exc:
                utils.print_error(f"{alias_name} via {surface_label} - ERROR: {exc}\n")


---
## ✅ What this notebook validated

1. **Provisioned 3 non-Foundry backends** with simple API-key auth:
   - `aws-bedrock` (Bedrock long-lived API key, Bearer)
   - `gemini` (`x-goog-api-key`)
   - `anthropic` (`x-api-key` + `anthropic-version`)
2. **Provisioned 2 OpenAI-compat passthroughs** so Bedrock and Gemini also serve `chat/completions` over both LLM API surfaces:
   - `aws-bedrock-mantle` and `gemini-openai`
3. **Validated routing across both LLM API surfaces** (URLs are `{gateway}` + APIM API URL suffix + operation):
   - **Universal LLM API** `/models/chat/completions` — fans out across Foundry / Bedrock-Mantle / Gemini-OpenAI
   - **Unified AI API** `/unified-ai/v1/chat/completions` — same fan-out via the `openai-compat` api-type
   - **Unified AI API** `/unified-ai/bedrock/...` — native Bedrock Converse (no body translation)
   - **Unified AI API** `/unified-ai/gemini/...` — native Gemini `generateContent`
   - **Unified AI API** `/unified-ai/claude/...` — native Anthropic Messages
4. **Verified `compatible-pool-types` filter** prevents native pools from leaking onto the
   OpenAI-compat surface (and vice-versa) even when model names overlap (no `-openai`
   suffix tricks needed any more).
5. **Validated three model-alias scenarios** (each opt-in, built only when its members exist):
   - **A. `foundry-weighted-mix`** — single-cloud weighted load-balance across Microsoft Foundry models. Same OpenAI `/v1/chat/completions` spec for every member; weights drive the random-by-weight pick on each call.
   - **B. `multi-cloud-openai`** — cross-cloud priority routing with transparent fallback across Foundry, Bedrock-Mantle, and Gemini-OpenAI. All members share the same OpenAI `/v1/chat/completions` spec, so the gateway forwards the call as-is and only rewrites the body's `model` field.
   - **C. `multi-cloud-claude`** — native Anthropic `/v1/messages` spec. Today the alias members are Anthropic API-key backends only; the same alias definition becomes truly multi-cloud once the future `aws-bedrock-anthropic-passthrough` and `foundry-anthropic-passthrough` backend types ship (no client-side change required).

### Phase scope: same-API-spec routing only

The current implementation **does not translate between API protocols**. Every alias must
front backends that share the same wire-level contract (path + body shape + auth model):

- Mixing `ai-foundry` with `aws-bedrock-mantle` and `gemini-openai` works ✅ — all three speak OpenAI `/v1/chat/completions`.
- Mixing `anthropic` (Anthropic Messages) with `aws-bedrock` (Bedrock Converse) ❌ — different request/response shapes; would need protocol translation.
- Mixing `gemini` (`generateContent`) with anything else ❌ — different schema.

A future phase will add **protocol-passthrough backend types** (e.g. an
`aws-bedrock-anthropic-passthrough` that exposes Bedrock's hosted Claude under the
Anthropic Messages spec, and a `foundry-anthropic-passthrough` for Foundry-hosted
Claude). When those land, scenario C extends across clouds with no caller-side change.

### Gateway changes that ship with this scenario

| File | Change |
|---|---|
| `bicep/infra/llm-backend-onboarding/modules/llm-backends.bicep` | New `gemini` / `anthropic` backendTypes. Auth headers configured **natively on the APIM Backend resource** via `credentials.header` — APIM substitutes `{{namedValueKey}}` to the named value's secret (Key Vault reference or inline) at backend create/update time. Header value = bare `{{namedValueKey}}` only (no concatenation), so the secret value must contain the **complete header value** (e.g. `Bearer sk-...` for Bearer auth, raw key otherwise). |
| `bicep/infra/llm-backend-onboarding/modules/llm-policy-fragments.bicep` | New `anthropic-version` named value + `anthropicVersion` parameter. Generates **alias virtual-pool entries** inside the `backendPools` JArray — each alias becomes a first-class entry with `isAlias=true`, `aliasName`, `strategy`, `weights`, and per-member resolved poolName/poolType/authType. |
| `bicep/infra/modules/apim/policies/frag-set-target-backend-pool.xml` (and dup) | Alias-aware short-circuit detects alias entries in `backendPools`, applies `compatiblePoolTypes` filter to alias members, picks one by `strategy`, sets all targeting variables (`is-alias`, `original-model-alias`, `requestedModel`, `targetBackendPool`, ...), exposes `alias-fallback-members` for the API policy retry block, and **rewrites the JSON body's `model` field to the resolved member** (so the body rewrite is co-located with the resolution and doesn't depend on fragment include order). |
| `bicep/infra/modules/apim/policies/frag-resolve-model-alias.xml` (and dup) | Slimmed to a body-rewrite-only no-op shim (the heavy lifting moved into `set-target-backend-pool`). Kept so legacy API policy include sites remain valid. |
| `bicep/infra/modules/apim/policies/universal-llm-api-policy-v2.xml` + `unified-ai-api-policy.xml` + `azure-open-ai-api-policy.xml` | Alias-aware retry block walks `alias-fallback-members` on 429/5xx for transparent same-spec fallback. Universal LLM and Azure OpenAI now have parity with Unified AI for cross-member retry. |
| `bicep/infra/modules/apim/policies/frag-metadata-config.xml` (and dup) | Native-prefix api-types (`bedrock-native`, `gemini-native`, `claude-native`); `compatible-pool-types` filter on every api-type that needs it; `backend-path-templates` for `anthropic` and existing OpenAI-compat backend types |
| `bicep/infra/modules/apim/policies/frag-request-processor.xml` | Reads `compatible-pool-types` from api-type config, exposes as `compatiblePoolTypes` variable. Model extraction extended for `bedrock-native` (`/model/{id}/...`) and `gemini-native` (`/models/{id}:...`) paths |
| `bicep/infra/modules/apim/policies/frag-path-builder.xml` | New prefix-strip branch for `bedrock-native` + `gemini-native`; fixed `/v1/messages` rewrite for `claude-native` |
| `bicep/infra/modules/apim/policies/frag-set-backend-authorization.xml` (and dup) | Now a **no-op for all `api-key-*` types** — those headers are set on the Backend resource. Only `aws-sigv4` (which needs runtime request signing) and `managed-identity` keep policy-side logic. |

### Caveats and design notes

- **Aliases are virtual policy-level pools** — APIM cannot put pools inside Backend Pool resources, so each alias is materialised as a JObject entry inside the `backendPools` JArray that `set-target-backend-pool` walks. Cross-member fallback is handled by the API policy `<retry>` block (one hop per remaining member).
- **Pre-stream fallback only** — once the response stream has started the body is committed and the gateway cannot retry against another alias member.
- **Each backend has its own `authConfig`** (separate named-value key + Key Vault secret). Do not share named values across backends with different header conventions, because the secret value must literally match the header APIM will send (e.g. `Bearer sk-...` for `Authorization`, but raw `sk-...` for `x-goog-api-key`).
- **Bedrock native uses `api-key-bearer`**, not AWS SigV4, in this notebook. The accelerator still ships full SigV4 support — set `authType: "aws-sigv4"` and supply `awsAccessKey` / `awsSecretKey` / `awsRegion` parameters when you need it.
- **Anthropic has no OpenAI-compat surface** in this notebook. To expose Claude over `chat/completions`, point an `aws-bedrock-mantle` backend at api.anthropic.com with a Bearer-prefixed secret — Anthropic also publishes an OpenAI-compatible endpoint that accepts Bearer auth.
- **No model name suffixes**: with the `compatible-pool-types` filter in place, the same model name can be shared by a native pool and an OpenAI-compat pool — the gateway picks the correct pool based on the inbound api-type, not on the model name.
- **Key Vault references for credentials**: prefer `keyVaultSecretUri` over `secretValue`. The Bicep module creates the named value as a Key Vault reference when the URI is set, so secrets stay in Key Vault, are auditable, and can be rotated without redeploying APIM.
